In [1]:
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import numpy as np

In [2]:
test_df = pd.read_csv('stored_results/Results_Logreg_No_GS.csv')
test_df

,Player,Hall of Fame,Prediction,Prediction Correct,Prediction Confidence
0,Cy Young,1,1,1,0.999999
1,Pud Galvin,1,1,1,0.618978
2,Walter Johnson,1,1,1,0.999999
3,Phil Niekro,1,1,1,0.918764
4,Nolan Ryan,1,1,1,0.998704
...,...,...,...,...,...
1210,Charlie Robertson,0,0,1,0.999562
1211,Sammy Ellis,0,0,1,0.993309
1212,Lil Stoner,0,0,1,0.999577
1213,Johnny Humphries,0,0,1,0.999019


In [5]:
# Subject to change if more models are tried later. Refer to Part 4a for full list.
model_files = {
    "Logistic Regression": "stored_results/Results_Logreg_No_GS.csv",
    "Decision Tree (GS for F1, Stratified KFold)": "stored_results/Results_DTree_GS_Best_F1_Stratified_KFold.csv",
    "Random Forest (GS for Recall, Stratified KFold)": "stored_results/Results_RFC_GS_Best_Recall_Stratified_KFold.csv",
    "Extra Trees (GS for Recall, Standard KFold)": "stored_results/Results_ETC_GS_Best_Recall_Standard_KFold.csv",
    "Gradient Booster (GS for Recall, Standard KFold)": "stored_results/Results_GBC_GS_Best_Recall_Standard_KFold.csv",
    "Support Vector (no GS, Stratified KFold)": "stored_results/Results_SVC_no_GS_Stratified_KFold.csv",
    "Neural Network": "stored_results/Results_Neural_Network_1.csv",
}

In [6]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
)
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, roc_auc_score
)

model_stats = []



# Define penalty weights
alpha, beta = 2, 2




for model_name, file_path in model_files.items():
    df = pd.read_csv(file_path)
    y_true = df['Hall of Fame']
    y_pred = df['Prediction']

    # Confirm binary labels
    labels = sorted(list(set(y_true) | set(y_pred)))
    print(f"{model_name} labels: {labels}")  # sanity check

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # Optional: if you have probabilities
    y_proba = df['Prediction Confidence'] if 'y_proba' in df.columns else None


    # Use probabilities if available
    if y_proba is not None:
        roc_auc = roc_auc_score(y_true, y_proba)
    else:
        roc_auc = roc_auc_score(y_true, y_pred)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)




    # Compute adjusted prediction score if probabilities are available

    
    if 'Prediction Confidence' in df.columns:
        p = df['Prediction Confidence']
        df['Adjusted Prediction Score'] = np.where(
            df['Prediction Correct'],
            p**alpha,
            -beta * p**alpha
        )
        mean_adj_score = df['Adjusted Prediction Score'].mean()
    else:
        mean_adj_score = np.nan  # no confidence info




    

    model_stats.append({
        "Model": model_name,
        "True Negatives": tn,
        "False Positives": fp,
        "False Negatives": fn,
        "True Positives": tp,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": roc_auc,
        "Mean Adjusted Prediction Score": mean_adj_score
    })


results_df_unweighed = pd.DataFrame(model_stats)

results_df_unweighed

Logistic Regression labels: [0, 1]
Decision Tree (GS for F1, Stratified KFold) labels: [0, 1]
Random Forest (GS for Recall, Stratified KFold) labels: [0, 1]
Extra Trees (GS for Recall, Standard KFold) labels: [0, 1]
Gradient Booster (GS for Recall, Standard KFold) labels: [0, 1]
Support Vector (no GS, Stratified KFold) labels: [0, 1]
Neural Network labels: [0, 1]


,Model,True Negatives,False Positives,False Negatives,True Positives,Accuracy,Precision,Recall,F1 Score,ROC AUC,Mean Adjusted Prediction Score
0,Logistic Regression,1117,16,34,48,0.958848,0.750000,0.585366,0.657534,0.785622,0.868560
1,"Decision Tree (GS for F1, Stratified KFold)",1101,32,32,50,0.947325,0.609756,0.609756,0.609756,0.790756,0.837889
2,"Random Forest (GS for Recall, Stratified KFold)",1123,10,31,51,0.966255,0.836066,0.621951,0.713287,0.806563,0.873379
3,"Extra Trees (GS for Recall, Standard KFold)",1110,23,22,60,0.962963,0.722892,0.731707,0.727273,0.855704,0.849992
4,"Gradient Booster (GS for Recall, Standard KFold)",1114,19,29,53,0.960494,0.736111,0.646341,0.688312,0.814786,0.884231
5,"Support Vector (no GS, Stratified KFold)",1126,7,54,28,0.949794,0.800000,0.341463,0.478632,0.667643,0.789771
6,Neural Network,1117,16,26,56,0.965432,0.777778,0.682927,0.727273,0.834403,0.018855
